In [8]:
%pip install requests

import requests
import os
import re
import json
import argparse
from datetime import datetime
from urllib.parse import urlparse


# ----------------------------
# Extraction helpers
# ----------------------------

IP_RE = re.compile(r"\b(?:(?:25[0-5]|2[0-4]\d|1?\d?\d)\.){3}(?:25[0-5]|2[0-4]\d|1?\d?\d)\b")
URL_RE = re.compile(r"https?://[^\s<>\")]+", re.IGNORECASE)
DOMAIN_RE = re.compile(r"\b(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,}\b", re.IGNORECASE)


def extract_ips(text: str) -> list[str]:
    return sorted(set(IP_RE.findall(text)))


def normalize_url(u: str) -> str:
    # Remove trailing punctuation common in copy/paste
    return u.rstrip(").,;!?:]'\"")


def extract_urls(text: str) -> list[str]:
    urls = [normalize_url(u) for u in URL_RE.findall(text)]
    # Deduplicate while keeping stable order
    seen = set()
    out = []
    for u in urls:
        if u not in seen:
            seen.add(u)
            out.append(u)
    return out


def extract_domains(text: str) -> list[str]:
    domains = set(DOMAIN_RE.findall(text))

    # Remove domains that are just parts of URLs already captured (still fine to keep),
    # but keep it simple: dedupe only.
    return sorted(domains)


def domains_from_urls(urls: list[str]) -> list[str]:
    out = set()
    for u in urls:
        try:
            host = urlparse(u).hostname
            if host:
                out.add(host.lower())
        except Exception:
            pass
    return sorted(out)


# ----------------------------
# Threat Intel clients (optional)
# ----------------------------

def vt_headers() -> dict:
    api_key = os.getenv("VT_API_KEY", "").strip()
    if not api_key:
        return {}
    return {"x-apikey": api_key}


def abuse_headers() -> dict:
    api_key = os.getenv("ABUSEIPDB_API_KEY", "").strip()
    if not api_key:
        return {}
    return {"Key": api_key, "Accept": "application/json"}


def vt_url_id(url: str) -> str:
    # VirusTotal expects URL id as base64url of the URL without padding.
    import base64
    b = base64.urlsafe_b64encode(url.encode("utf-8")).decode("ascii")
    return b.strip("=")


def vt_lookup_ip(ip: str) -> dict | None:
    h = vt_headers()
    if not h:
        return None
    r = requests.get(f"https://www.virustotal.com/api/v3/ip_addresses/{ip}", headers=h, timeout=15)
    if r.status_code != 200:
        return {"error": f"VT IP lookup failed ({r.status_code})", "body": safe_json(r)}
    return safe_json(r)


def vt_lookup_domain(domain: str) -> dict | None:
    h = vt_headers()
    if not h:
        return None
    r = requests.get(f"https://www.virustotal.com/api/v3/domains/{domain}", headers=h, timeout=15)
    if r.status_code != 200:
        return {"error": f"VT domain lookup failed ({r.status_code})", "body": safe_json(r)}
    return safe_json(r)


def vt_lookup_url(url: str) -> dict | None:
    h = vt_headers()
    if not h:
        return None
    url_id = vt_url_id(url)
    r = requests.get(f"https://www.virustotal.com/api/v3/urls/{url_id}", headers=h, timeout=15)
    if r.status_code != 200:
        return {"error": f"VT URL lookup failed ({r.status_code})", "body": safe_json(r)}
    return safe_json(r)


def abuseipdb_check(ip: str) -> dict | None:
    h = abuse_headers()
    if not h:
        return None
    params = {"ipAddress": ip, "maxAgeInDays": 90, "verbose": ""}
    r = requests.get("https://api.abuseipdb.com/api/v2/check", headers=h, params=params, timeout=15)
    if r.status_code != 200:
        return {"error": f"AbuseIPDB lookup failed ({r.status_code})", "body": safe_json(r)}
    return safe_json(r)


def safe_json(resp: requests.Response) -> dict:
    try:
        return resp.json()
    except Exception:
        return {"raw_text": resp.text[:2000]}


# ----------------------------
# Link builders (always available, even without API keys)
# ----------------------------

def build_links(ips: list[str], domains: list[str], urls: list[str]) -> dict:
    links = {"ips": {}, "domains": {}, "urls": {}}

    for ip in ips:
        links["ips"][ip] = {
            "abuseipdb": f"https://www.abuseipdb.com/check/{ip}",
            "virustotal": f"https://www.virustotal.com/gui/ip-address/{ip}",
            "whois": f"https://who.is/whois-ip/ip-address/{ip}",
        }

    for d in domains:
        links["domains"][d] = {
            "virustotal": f"https://www.virustotal.com/gui/domain/{d}",
            "whois": f"https://who.is/whois/{d}",
            "urlscan_search": f"https://urlscan.io/search/#domain:{d}",
        }

    for u in urls:
        links["urls"][u] = {
            "virustotal": f"https://www.virustotal.com/gui/url/{vt_url_id(u)}",
            "urlscan_search": f"https://urlscan.io/search/#page.url:{u}",
        }

    return links


# ----------------------------
# Reporting
# ----------------------------

def summarize_vt(obj: dict) -> str:
    # Best-effort extraction from VT v3 responses
    try:
        stats = obj["data"]["attributes"]["last_analysis_stats"]
        return f"VT stats: harmless={stats.get('harmless')} malicious={stats.get('malicious')} suspicious={stats.get('suspicious')} undetected={stats.get('undetected')}"
    except Exception:
        return "VT stats: (unavailable)"


def summarize_abuse(obj: dict) -> str:
    try:
        data = obj["data"]
        return f"AbuseIPDB: score={data.get('abuseConfidenceScore')} reports={data.get('totalReports')} country={data.get('countryCode')}"
    except Exception:
        return "AbuseIPDB: (unavailable)"


def write_markdown_report(out_path: str, extracted: dict, links: dict, intel: dict):
    now = datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S UTC")
    lines = []
    lines.append(f"# Phishing Triage Report\n")
    lines.append(f"- Generated: {now}\n")

    lines.append("## Extracted Indicators\n")
    lines.append(f"- IPs ({len(extracted['ips'])}): {', '.join(extracted['ips']) if extracted['ips'] else 'None'}")
    lines.append(f"- Domains ({len(extracted['domains'])}): {', '.join(extracted['domains']) if extracted['domains'] else 'None'}")
    lines.append(f"- URLs ({len(extracted['urls'])}): {len(extracted['urls']) if extracted['urls'] else 0}\n")

    if extracted["urls"]:
        lines.append("### URLs\n")
        for u in extracted["urls"]:
            lines.append(f"- {u}")
        lines.append("")

    lines.append("## Quick Links\n")

    if extracted["ips"]:
        lines.append("### IP Links\n")
        for ip, l in links["ips"].items():
            lines.append(f"- **{ip}**: [AbuseIPDB]({l['abuseipdb']}) | [VirusTotal]({l['virustotal']}) | [WHOIS]({l['whois']})")
        lines.append("")

    if extracted["domains"]:
        lines.append("### Domain Links\n")
        for d, l in links["domains"].items():
            lines.append(f"- **{d}**: [VirusTotal]({l['virustotal']}) | [WHOIS]({l['whois']}) | [urlscan search]({l['urlscan_search']})")
        lines.append("")

    if extracted["urls"]:
        lines.append("### URL Links\n")
        for u, l in links["urls"].items():
            lines.append(f"- **URL**: [VirusTotal]({l['virustotal']}) | [urlscan search]({l['urlscan_search']})")
            lines.append(f"  - {u}")
        lines.append("")

    lines.append("## Threat Intel (API Lookups)\n")
    lines.append("_If API keys are not configured, this section will be empty._\n")

    # IP intel
    if intel.get("ips"):
        lines.append("### IP Intelligence\n")
        for ip, data in intel["ips"].items():
            lines.append(f"- **{ip}**")
            if "abuseipdb" in data and data["abuseipdb"]:
                lines.append(f"  - {summarize_abuse(data['abuseipdb'])}")
            if "virustotal" in data and data["virustotal"]:
                lines.append(f"  - {summarize_vt(data['virustotal'])}")
        lines.append("")

    # Domain intel
    if intel.get("domains"):
        lines.append("### Domain Intelligence\n")
        for d, data in intel["domains"].items():
            lines.append(f"- **{d}**")
            if "virustotal" in data and data["virustotal"]:
                lines.append(f"  - {summarize_vt(data['virustotal'])}")
        lines.append("")

    # URL intel
    if intel.get("urls"):
        lines.append("### URL Intelligence\n")
        for u, data in intel["urls"].items():
            lines.append(f"- **URL**: {u}")
            if "virustotal" in data and data["virustotal"]:
                lines.append(f"  - {summarize_vt(data['virustotal'])}")
        lines.append("")

    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))


def main():
    parser = argparse.ArgumentParser(description="Phishing triage helper: extract indicators, generate links, and optionally query VT/AbuseIPDB.")
    parser.add_argument("--input", "-i", required=True, help="Path to a text file containing email headers (and optional body).")
    parser.add_argument("--out", "-o", default=None, help="Output report path (.md). Default: reports/triage_<timestamp>.md")
    args = parser.parse_args()

    with open(args.input, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    ips = extract_ips(text)
    urls = extract_urls(text)
    domains = sorted(set(extract_domains(text) + domains_from_urls(urls)))

    extracted = {"ips": ips, "urls": urls, "domains": domains}
    links = build_links(ips, domains, urls)

    # Optional intel lookups
    intel = {"ips": {}, "domains": {}, "urls": {}}

    vt_enabled = bool(os.getenv("VT_API_KEY", "").strip())
    abuse_enabled = bool(os.getenv("ABUSEIPDB_API_KEY", "").strip())

    for ip in ips:
        intel["ips"][ip] = {
            "virustotal": vt_lookup_ip(ip) if vt_enabled else None,
            "abuseipdb": abuseipdb_check(ip) if abuse_enabled else None,
        }

    for d in domains:
        intel["domains"][d] = {
            "virustotal": vt_lookup_domain(d) if vt_enabled else None,
        }

    for u in urls:
        intel["urls"][u] = {
            "virustotal": vt_lookup_url(u) if vt_enabled else None,
        }

    # Output
    ts = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
    out_path = args.out or f"reports/triage_{ts}.md"

    # Create reports dir if needed
    os.makedirs(os.path.dirname(out_path), exist_ok=True)

    write_markdown_report(out_path, extracted, links, intel)

    print(f"[+] Report written: {out_path}")
    print(f"[+] Extracted: ips={len(ips)} domains={len(domains)} urls={len(urls)}")
    if not vt_enabled:
        print("[!] VT_API_KEY not set; VirusTotal API lookups skipped (links still included).")
    if not abuse_enabled:
        print("[!] ABUSEIPDB_API_KEY not set; AbuseIPDB API lookups skipped (links still included).")


if __name__ == "__main__":
    main()
    

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip
usage: ipykernel_launcher.py [-h] --input INPUT [--out OUT]
ipykernel_launcher.py: error: the following arguments are required: --input/-i


SystemExit: 2